# Reproduction — *The Power of Noise* (Cuconasu et al., SIGIR 2024)

Small-scale reproduction on NQ-open with Llama-2-7B-Chat (4-bit). See `PROTOCOLS.md`, Protocol 1.

**How to work in this notebook**
- `CONFIG["debug"] = True` runs everything on ~20 questions. Switch it off only when a section is validated.
- Cells marked `[TO VERIFY]` depend on something we must check in the authors' code or paper.
- Every experiment writes one `.jsonl` file (one line per question) and skips what already exists, so a crashed run can resume.

**Validation checkpoints (run in this order, compare with the paper)**

| Step | Expected (Llama2-7B, paper) | If far off |
| --- | --- | --- |
| Closed-book | ~0.11 | Prompt format or `contains_answer` is wrong |
| Gold only | ~0.56 | Same, or gold passage not the one expected |
| Gold + distracting (Table 1) | 0.43 / 0.40 / 0.38 for 1 / 2 / 4 | Distractor selection |
| Gold + random (Table 2) | see paper | Random sampling, position |
| Retrieved + random (Table 3) | see `REFERENCE` below | Context order, retrieval files |

**Expected sampling variability**

| Questions | Gold only (~0.56) | Closed-book (~0.11) |
| ---: | ---: | ---: |
| 100 | ±10 percentage points | ±6 percentage points |
| 500 | ±4–5 percentage points | ±3 percentage points |

## 0. Setup

Install (once, in the environment): `torch` (a recent build supporting the RTX 5060 or your GPU), `transformers`, `accelerate`, `bitsandbytes`, `pandas`, `scipy`, `tqdm`, `requests`, `gdown`, `ijson`.
Llama-2 is gated: accept the license on its Hugging Face page, then log in with `huggingface-cli login`.

In [5]:
import json, os, random, re, time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

import torch
import transformers

import shutil
import requests
import json, pickle, ijson
import regex, string, hashlib, unicodedata

from dotenv import load_dotenv, find_dotenv

# find_dotenv searches the current folder, then the parent folders, so a .env at the repository root is found
# usecwd=True is needed in a notebook (there is no __file__)
load_dotenv(find_dotenv(usecwd=True))
assert os.environ.get("HF_TOKEN"), ".env not found or HF_TOKEN missing"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"   # hides the Windows symlink warning

In [7]:
CONFIG = dict(
    seed=42,                     # only to draw the question sample (random passages are stored in the data files)
    model_id="meta-llama/Llama-2-7b-chat-hf",
    n_questions=500,             # per split (paper: 2,889 test / 10,000 train sample)
    debug=True,
    n_debug=20,
    max_new_tokens=15,           # authors' default
    repetition_penalty=1.1,      # authors' `LLM.generate` uses it (not in the paper)
    batch_size=4,
    max_context_tokens=4096,     # authors' model_max_length for Llama-2
    skip_margin=2,               # authors skip prompts with >= max_context_tokens - 2 tokens
    normalize_query=True,        # authors pass do_normalize_query=True
    use_chat_template=False,     # authors feed the raw text, without [INST] tags
    data_dir=Path("data"),
    output_dir=Path("outputs/nq"),
)

# Create directories if they don't exist
CONFIG["data_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)

N = CONFIG["n_debug"] if CONFIG["debug"] else CONFIG["n_questions"]
rng = random.Random(CONFIG["seed"])

In [8]:
# Paper values (Llama2-7B) to compare with. Table 3 keys: (retriever, n_retrieved, n_random)
REFERENCE = {
    "closed_book": 0.1123,
    "gold_only": 0.5642,
    "table1_near": {0: 0.5642, 1: 0.4283, 2: 0.3974, 4: 0.3795},   # n distracting
    "table3": {
        ("contriever", 1, 0): 0.1620, ("contriever", 1, 3): 0.1301, ("contriever", 1, 10): 0.1796,
        ("contriever", 4, 0): 0.1866, ("contriever", 4, 3): 0.2316, ("contriever", 4, 10): 0.2502,
        ("bm25", 1, 0): 0.2008, ("bm25", 1, 3): 0.1568, ("bm25", 1, 10): 0.2108,
        ("bm25", 4, 0): 0.2028, ("bm25", 4, 3): 0.2520, ("bm25", 4, 10): 0.2835,
    },
    # Table 2 (gold + random, near) values from the paper for 2 and 4 random documents (only Llama2 results)
    "table2": {
        ("far", 2): 0.3776, ("far", 4): 0.3109,
        ("mid", 2): 0.3928, ("mid", 4): 0.3998,
        ("near", 2): 0.5032, ("near", 4): 0.5221,
    },
}

In [9]:
def check_environment(tiny_model="facebook/opt-125m") -> None:
    """Print GPU info and library versions, then load a tiny model in 4 bit with the authors'
    quantization settings (nf4, double quant, bfloat16 compute) and generate a few tokens."""
    import bitsandbytes as bnb
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    print("torch", torch.__version__, "| transformers", transformers.__version__,
          "| bitsandbytes", bnb.__version__, "| CUDA (torch)", torch.version.cuda)
    assert torch.cuda.is_available(), "No CUDA GPU visible to torch"
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} | capability {torch.cuda.get_device_capability(0)} | "
          f"free {free/1e9:.1f} GB / {total/1e9:.1f} GB | bf16 supported: {torch.cuda.is_bf16_supported()}")
    
    print("arch list:", torch.cuda.get_arch_list())
    x = torch.randn(1000, 1000, device="cuda", dtype=torch.bfloat16)
    print("matmul on GPU works:", (x @ x).shape)
    
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tok = AutoTokenizer.from_pretrained(tiny_model, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(tiny_model, quantization_config=bnb_config,
                                                 torch_dtype=torch.bfloat16, device_map="auto")
    inputs = tok(["The capital of France is"], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, do_sample=False, max_new_tokens=5, repetition_penalty=1.1)
    print("4-bit generation works:", repr(tok.decode(out[0], skip_special_tokens=True)))
    del model; torch.cuda.empty_cache()

check_environment()

W1114 23:43:29.840000 25444 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


torch 2.15.0+cu132 | transformers 5.23.0 | bitsandbytes 0.50.2 | CUDA (torch) 13.2
GPU: NVIDIA GeForce RTX 5060 Laptop GPU | capability (12, 0) | free 7.4 GB / 8.5 GB | bf16 supported: True
arch list: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
matmul on GPU works: torch.Size([1000, 1000])


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 236.05it/s]


4-bit generation works: 'The capital of France is the capital of the French'


## 0b. Download the data

One call downloads only what we need into `data/`, from the authors' GitHub repository (questions, search results, mappings) and from their shared Google Drive (corpus subsets). It skips files that are already valid, so it is safe to re-run.

| Stage | Used for | Approx. size |
| --- | --- | --- |
| `base` | closed-book, gold-only, Table 2 (gold + random). Includes the `random_at60` subset, which holds the text **and title** of the gold passages (the question dataset file has no title) | ~395 MB |
| `table1` | gold + distracting (Contriever) | ~650 MB |
| `table3` | retrieved + random on the test split | 1,010 MB + 1,180 MB |
| `adore` | optional ADORE row of Table 1 | not in the defaults |

**Experiments:**

| Experiment | Stage | Context given to the LLM | Paper value (Llama2-7B) | Generations | Files (data/) | Size |
| --- | --- | --- | ---: | ---: | --- | --- |
| Closed-book | base | Question only and possible answers: no document, no retriever | 0.11 | 500 | `10k_train_dataset.json` | 7 MB |
| Gold only | base | Question, possible answers and the gold passage alone (oracle), no retriever | 0.56 | 500 | `10k_train_dataset.json` (questions, possible answers, gold passage), `mappings/subset_to_full_random_at60_in_corpus.pkl` and `processed/corpus_with_random_at60.json` subset + mapping (title and text of the gold) | 392 MB |
| Table 2: gold + random | base | Gold passage next to the question, preceded by 2 or 4 random passages | 0.50 (2), 0.52 (4) | 2 × 500 = 1,000 | `10k_train_dataset.json`, `10k_random_results_at60.pkl` (60 ids of random passages for each query) | 395 MB |
| Table 1: gold + distracting | table1 | Gold next to the question, preceded by 1, 2 or 4 top-ranked Contriever passages that do not contain the answer | 0.43 / 0.40 / 0.38 | 3 × 500 = 1,500 | `10k_train_dataset.json` (question, gold, answers), `contriever_search_results_at150.pkl` (top-150 for each question), `mappings/subset_to_full_contriever_at150_in_corpus.pkl` (mapping), `processed/corpus_with_contriever_at150.json` (subset corpus) | 635 MB |
| Table 3: retrieved + random | table3 | No gold injected. The top-1 or top-4 passages retrieved by Contriever or BM25 next to the question, preceded by 0, 3 or 10 random passages | With 4 retrieved, 0 / 3 / 10 random: Contriever 0.19 / 0.23 / 0.25, BM25 0.20 / 0.25 / 0.28 | 12 configurations × 500 = 6,000 | `test_dataset.json` (2 889 question), Contriever and BM25 test search results, 2 test subsets + mappings (`mappings/subset_to_full_test_random_contriever_in_corpus.pkl`mapping, `contriever_test_search_results_at150.pkl`top-150, `processed/test_corpus_with_random_contriever.json` subset corpus, same for BM25) + random passages (`10k_other_random_results_at60.pkl`) | ~2240 MB |
| ADORE (optional) | adore | Same as Table 1, distractors selected by ADORE | 0.41 / 0.38 / 0.36 | 3 × 500 = 1,500 | `adore_*` files | Not checked |


Total: 9,500 generations with the default stages, 11,000 with ADORE.

It never downloads the 12 GB `corpus.json`, `corpus.zip` or `train_dataset.json`. To start light: `ensure_data(stages=["base"])`; preview without downloading: `ensure_data(dry_run=True)`. If the Drive refuses (download quota), the function prints which files to put by hand and where.



In [10]:
GITHUB_RAW = "https://raw.githubusercontent.com/florin-git/The-Power-of-Noise/main/data"
# Shared folder given in the authors' README
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1MfR7mJ76tyVpjbMwUkMVbEjOQKpdL-Lq"
DATA_DIR = Path(CONFIG["data_dir"])

# stage -> [(path relative to data/, where to look in order, approximate size in MB or None)]
# Only what we need: never the 12 GB corpus.json, corpus.zip or train_dataset.json.
DATA_STAGES = {
    "base": [                                        # closed-book, gold-only and Table 2 (gold + random)
        ("10k_train_dataset.json", ("github", "drive"), 7.3),
        ("10k_random_results_at60.pkl", ("github", "drive"), 3.1),            # 60 random passages / question
        ("mappings/subset_to_full_random_at60_in_corpus.pkl", ("github", "drive"), 5.9),
        ("processed/corpus_with_random_at60.json", ("drive",), 379),           # text + title of gold and random
    ],
    "table1": [                                      # gold + distracting (Contriever, train sample)
        ("contriever_search_results_at150.pkl", ("github", "drive"), 37.8),
        ("mappings/subset_to_full_contriever_at150_in_corpus.pkl", ("github", "drive"), 9.3),
        ("processed/corpus_with_contriever_at150.json", ("drive",), 590),
    ],
    "table3": [                                      # retrieved + random (test split)
        ("test_dataset.json", ("github", "drive"), 2.1),
        ("contriever_test_search_results_at150.pkl", ("github", "drive"), 10.9),
        ("bm25_test_search_results_at250.pkl", ("github", "drive"), 10.2),
        ("mappings/subset_to_full_test_random_contriever_in_corpus.pkl", ("github", "drive"), 15.2),
        ("mappings/subset_to_full_test_random_bm25_in_corpus.pkl", ("github", "drive"), 17.8),
        ("processed/test_corpus_with_random_contriever.json", ("drive",), 1010),
        ("processed/test_corpus_with_random_bm25.json", ("drive",), 1180),
        ("10k_other_random_results_at60.pkl", ("github", "drive"), 3.1), # random passages for test questions
    ],
    "adore": [                                       # optional: ADORE row of Table 1
        ("adore_search_results_at200.pkl", ("github", "drive"), 50.4),
        ("mappings/subset_to_full_adore_at200_in_corpus.pkl", ("github", "drive"), 12.7),
        ("processed/corpus_with_adore_at200.json", ("drive",), None),
    ],
}
DEFAULT_STAGES = ("base", "table1", "table3")

# Local paths, keyed by (split, name), for section 1
DATA_FILES = {
    "questions": {"train": DATA_DIR / "10k_train_dataset.json", "test": DATA_DIR / "test_dataset.json"},
    "search": {
        ("train", "random"):     DATA_DIR / "10k_random_results_at60.pkl",
        ("train", "contriever"): DATA_DIR / "contriever_search_results_at150.pkl",
        ("test", "contriever"):  DATA_DIR / "contriever_test_search_results_at150.pkl",
        ("test", "bm25"):        DATA_DIR / "bm25_test_search_results_at250.pkl",
        ("train", "adore"):      DATA_DIR / "adore_search_results_at200.pkl",
        ("test", "random"):      DATA_DIR / "10k_other_random_results_at60.pkl",
    },
    "subset": {   # (corpus subset JSON, table: position in the subset -> global index)
        ("train", "random"):     (DATA_DIR / "processed/corpus_with_random_at60.json",
                                  DATA_DIR / "mappings/subset_to_full_random_at60_in_corpus.pkl"),
        ("train", "contriever"): (DATA_DIR / "processed/corpus_with_contriever_at150.json",
                                  DATA_DIR / "mappings/subset_to_full_contriever_at150_in_corpus.pkl"),
        ("test", "contriever"):  (DATA_DIR / "processed/test_corpus_with_random_contriever.json",
                                  DATA_DIR / "mappings/subset_to_full_test_random_contriever_in_corpus.pkl"),
        ("test", "bm25"):        (DATA_DIR / "processed/test_corpus_with_random_bm25.json",
                                  DATA_DIR / "mappings/subset_to_full_test_random_bm25_in_corpus.pkl"),
        ("train", "adore"):      (DATA_DIR / "processed/corpus_with_adore_at200.json",
                                  DATA_DIR / "mappings/subset_to_full_adore_at200_in_corpus.pkl"),
    },
}


def _looks_valid(path: Path) -> bool:
    """True if the file exists and looks like real data (not empty, not an HTML error page,
    not a Git LFS pointer; pickles start with 0x80 and JSON with [ or {)."""
    if not path.exists() or path.stat().st_size < 1000:
        return False
    with open(path, "rb") as f:
        head = f.read(64).lstrip()
    if head.startswith(b"version https://git-lfs") or head.lower().startswith((b"<!doctype", b"<html")):
        return False
    if path.suffix == ".pkl":
        return head[:1] == b"\x80"
    if path.suffix == ".json":
        return head[:1] in (b"[", b"{")
    return True


def _download_github(rel: str, dest: Path) -> None:
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(f"{GITHUB_RAW}/{rel}", stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp.replace(dest)


_DRIVE_LISTING = None

def _drive_listing():
    """List the files of the authors' Drive folder WITHOUT downloading anything."""
    global _DRIVE_LISTING
    if _DRIVE_LISTING is None:
        import gdown
        try:
            _DRIVE_LISTING = gdown.download_folder(url=DRIVE_FOLDER_URL, skip_download=True,
                                                   quiet=True, remaining_ok=True)
        except TypeError:   # recent gdown versions have no `remaining_ok`
            _DRIVE_LISTING = gdown.download_folder(url=DRIVE_FOLDER_URL, skip_download=True, quiet=True)
    return _DRIVE_LISTING


def _find_drive_id(rel: str) -> str:
    """Id of the Drive file whose path ends with `rel` (works whichever folder the link points to)."""
    want = tuple(rel.split("/"))
    hits = [f for f in _drive_listing()
            if tuple(f.path.replace("\\", "/").split("/"))[-len(want):] == want]
    if len(hits) != 1:
        raise FileNotFoundError(f"{rel}: {len(hits)} matches in the Drive folder")
    return hits[0].id


def _download_drive(rel: str, dest: Path) -> None:
    import gdown
    tmp = dest.with_name(dest.name + ".part")
    out = gdown.download(id=_find_drive_id(rel), output=str(tmp), quiet=False)
    if out is None:
        raise RuntimeError("gdown returned nothing (Drive download quota reached?)")
    Path(out).replace(dest)


def ensure_data(stages=DEFAULT_STAGES, dry_run: bool = False) -> None:
    """Download what is missing, check it, skip what is already valid (safe to re-run).
    Use dry_run=True to see the plan and the disk check without downloading."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    todo = [(stage, rel, src, mb) for stage in stages for rel, src, mb in DATA_STAGES[stage]
            if not _looks_valid(DATA_DIR / rel)]
    if not todo:
        print("All files are already present and valid.")
        return

    known_mb = sum(mb for *_, mb in todo if mb)
    free_mb = shutil.disk_usage(DATA_DIR).free / 1e6
    print(f"{len(todo)} file(s) to download, at least {known_mb:,.0f} MB "
          f"(some sizes unknown); free disk space: {free_mb:,.0f} MB")
    for stage, rel, src, mb in todo:
        print(f"  [{stage:10}] {rel}  ({'?' if mb is None else f'{mb:g}'} MB, from {src[0]})")
    if free_mb < 1.2 * known_mb:
        raise RuntimeError("Not enough free disk space for the data files.")
    if dry_run:
        return

    failed = []
    for stage, rel, sources, mb in todo:
        dest = DATA_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        ok = False
        for source in sources:
            try:
                print(f"-> {rel}  [{source}]")
                (_download_github if source == "github" else _download_drive)(rel, dest)
                ok = _looks_valid(dest)
                if ok:
                    break
                print("   downloaded file does not look valid, trying the next source")
                dest.unlink(missing_ok=True)
            except Exception as e:
                print(f"   failed: {type(e).__name__}: {str(e)[:150]}")
        if not ok:
            failed.append(rel)

    if failed:
        print("\nCould not download automatically (Drive quota, network, ...):")
        for rel in failed:
            print(f"  - {rel}\n    put it by hand in {DATA_DIR / rel}")
        print(f"  Folder: {DRIVE_FOLDER_URL}")
        raise RuntimeError(f"{len(failed)} file(s) missing")
    print("\nDone: all files downloaded and checked.")

In [11]:
ensure_data()   # downloads what is missing; prints the plan and checks disk space first

All files are already present and valid.


**Downloaded files**

| No. | File | Source | Size | Contents | Used for |
|---:|---|---|---:|---|---|
| 1 | `10k_train_dataset.json` | GitHub | 7 MB | 10,000 NQ-open questions: `example_id`, `question`, `answers` (accepted answers), `text` (gold passage), `idx_gold_in_corpus` (global index). No title. | Questions and answers for closed-book, gold-only, and Tables 1 and 2 |
| 2 | `10k_random_results_at60.pkl` | GitHub | 3 MB | For each question, 60 global indices of randomly sampled passages (never that question’s own gold passage (measured, 0 / 10,000)), followed by an integer whose meaning is unknown. | Table 2: gold + random passages |
| 3 | `mappings/subset_to_full_random_at60_in_corpus.pkl` | GitHub | 6 MB | Dictionary with 600,639 entries: subset position → global index. | Retrieve passage text for file 4 |
| 4 | `processed/corpus_with_random_at60.json` | Drive | 398 MB | Passages (title + text): the 10,000 gold passages and the random passages from file 2. | Gold title and text; text of random documents (Table 2) |
| 5 | `contriever_search_results_at150.pkl` | GitHub | 38 MB | For each question in the sample: global indices of the 150 highest-ranked passages from Contriever and their scores sorted in decreasing order (checked). | Table 1: select highly ranked distractors that do not contain the answer |
| 6 | `mappings/subset_to_full_contriever_at150_in_corpus.pkl` | GitHub | 9 MB | Position → global index mapping for file 7. | Retrieve passage text for file 7 |
| 7 | `processed/corpus_with_contriever_at150.json` | Drive | 618 MB | Passages (title + text) appearing in these rankings, plus the gold passages (to verify). | Text of distractors and gold passage (Table 1) |
| 8 | `test_dataset.json` | GitHub | 2 MB | 2,889 test-split questions, same fields as file 1 (checked). | Questions and answers for Table 3 |
| 9 | `contriever_test_search_results_at150.pkl` | GitHub | 11 MB | Contriever’s top 150 results for each test question. | Table 3: Contriever retriever |
| 10 | `bm25_test_search_results_at250.pkl` | GitHub | 10 MB | BM25’s top 250 results for each test question. | Table 3: BM25 retriever |
| 11 | `mappings/subset_to_full_test_random_contriever_in_corpus.pkl` | GitHub | 15 MB | Position → global index mapping for file 13. | Retrieve passage text for file 13 |
| 12 | `mappings/subset_to_full_test_random_bm25_in_corpus.pkl` | GitHub | 18 MB | Position → global index mapping for file 14. | Retrieve passage text for file 14 |
| 13 | `processed/test_corpus_with_random_contriever.json` | Drive | 1.01 GB | Passages (title + text) from Contriever’s top 150 test results, plus random passages (to verify). | Text of retrieved and random documents (Table 3, Contriever) |
| 14 | `processed/test_corpus_with_random_bm25.json` | Drive | 1.18 GB | Same as above, for BM25. | Text of retrieved and random documents (Table 3, BM25) |
| 15 | `10k_other_random_results_at60.pkl` | GitHub | 3.1 MB | 60 random ids per question | Used by Table 3 (test split) |

**Files not downloaded**

| File | Why it is omitted |
|---|---|
| `corpus.json`, `corpus.zip` | Full corpus (12 GB); the subsets are sufficient. |
| `train_dataset.json`, `dev_dataset.json` | Full NQ splits; we use the 10,000-question sample and the test split. |
| `mappings/full_to_subset_*` | Inverse mapping tables, which can be reconstructed. |
| `adore_*`, `corpus_with_adore_at200.json` | Optional ADORE row for Table 1 (`adore` stage). |
| `nonsense_random_results.pkl`, `reddit_*`, `corpus_with_random_50_words.pkl` | Table 4 (Reddit noise and random words), outside the scope. |
| `corpus_with_random_contriever.json` (without `test_`) | Usage not identified. |

## 1. Data

This section does two things: it documents what the authors' files contain (including a few
non-obvious facts we checked), and it provides the loading functions used by every experiment.
Nothing is loaded here: each experiment loads only what it needs and frees it afterwards
(8 GB of RAM; the Table 3 corpus files are 1 to 1.2 GB on disk).

### 1.1 What the files are

| Kind | Files | What it holds |
| --- | --- | --- |
| Questions | `10k_train_dataset.json`, `test_dataset.json` | NQ questions with `answers`, the gold passage `text` and its `idx_gold_in_corpus` (no title) |
| Search results | `*_search_results_*.pkl` | For each question, a tuple (passage ids, scores) from Contriever, BM25 or ADORE. Ids are strings in the Contriever files and integers in the random file. load_search_results casts all to `int` |
| Corpus subsets | `processed/*.json` | Only the passages that an experiment can use (title and text), stored as a list |
| Mappings | `mappings/subset_to_full_*.pkl` | Dictionary: position in the subset → id in the full 21,035,236-passage corpus |



### 1.2 Loading functions

In [12]:
def load_nq(path: str) -> list[dict]:
    """Load questions NQ as a list of dicts (question, answers, text, idx_gold_in_corpus...).
    """
    return json.load(open(path, encoding="utf-8"))

def load_search_results(path: Path) -> dict:
    """Load an authors' search-result file (pickle). 
    Return a list for each question of ids (int) corresponding to the top-k results.
    """
    res = pickle.load(open(path, "rb"))
    return [[int(i) for i in r[0]] for r in res]


def load_corpus_subset(corpus_path, mapping_path, needed_ids) -> dict[int, dict]:
    """Reads the large corpus file in a stream and retains only the requested passages.
    Parameters:
        - corpus_path: Path to the corpus file.
        - mapping_path: Path to the index mapping file.
        - needed_ids: IDs from the complete corpus.
    Returns {id: {“title”:str, “text”:str}}.
    """
    subset_to_full = pickle.load(open(mapping_path, "rb"))
    full_to_subset = {full: pos for pos, full in subset_to_full.items()} # ids in the full corpus -> positions in the subset
    wanted = {full_to_subset[i] for i in needed_ids}      # positions to read (ids in the subset)
    last = max(wanted)
    out = {}
    with open(corpus_path, "rb") as f:
        for pos, doc in enumerate(ijson.items(f, "item")):
            if pos in wanted:
                out[subset_to_full[pos]] = {"title": doc.get("title", ""), "text": doc["text"]}
            if pos >= last:
                break
    return out

### 1.3 Sanity checks

`check_data` is run on every (questions, search results, mapping) triple before any experiment. It stops with an error if something that the experiments rely on is wrong: files of different lengths, a missing field, a retrieved id that cannot be found in the subset (the passage could not be loaded), or a gold id missing where we expect it. It only reports, without stopping, what we cannot assume: duplicate ids in a ranking, scores that are not sorted, gold ids absent from a subset.

In [13]:
def check_data(name, questions, raw_results, mapping, expected_k=None,
               has_scores=True, require_gold_in_subset=False):
    """Check that questions, raw search results and a mapping can be used together.
    questions: list of dicts from load_nq; raw_results: the unpickled list of (ids, scores);
    mapping: dict subset position -> full corpus id. Raises AssertionError on a blocking problem."""
    problems, notes = [], []
    n = len(questions)

    # 1. Same number of questions and result lists (entry k of the results = question k)
    if len(raw_results) != n:
        problems.append(f"{n} questions but {len(raw_results)} result lists")

    # 2. Required fields and non-empty answers
    required = ("example_id", "question", "answers", "text", "idx_gold_in_corpus")
    missing_fields = [k for k in required if any(k not in q for q in questions)]
    if missing_fields:
        problems.append(f"missing fields in some questions: {missing_fields}")
    elif any(len(q["answers"]) == 0 for q in questions):
        problems.append("some questions have no answer")

    # 3. Every question has at least the expected number of ids (extra ids are harmless)
    # It was observed that some search results have more than the expected number of ids (e.g. 140 instead of 60 for the query 5445).
    lengths = [len(r[0]) for r in raw_results]
    if expected_k is not None and min(lengths) < expected_k:
        problems.append(f"some lists are shorter than {expected_k}: min length {min(lengths)}")
    if len(set(lengths)) != 1:
        notes.append(f"list lengths differ: {sorted(set(lengths))}; "
                     f"{sum(l != expected_k for l in lengths)} question(s) off")
    ks = set(lengths)

    # 4. Every retrieved id exists in the subset (otherwise the passage cannot be loaded)
    full_ids = set(mapping.values())
    retrieved = {int(i) for r in raw_results for i in r[0]}
    not_in_subset = retrieved - full_ids
    if not_in_subset:
        problems.append(f"{len(not_in_subset)} retrieved ids are not in the mapping")

    # 5. Gold passages in the subset (blocking only if the experiment needs them)
    gold_missing = {int(q["idx_gold_in_corpus"]) for q in questions} - full_ids
    if gold_missing:
        msg = f"{len(gold_missing)} gold ids are not in the subset"
        (problems if require_gold_in_subset else notes).append(msg)

    # 6. Non-blocking observations on the rankings
    dup = sum(len(set(r[0])) != len(r[0]) for r in raw_results)
    if dup:
        notes.append(f"{dup} questions have duplicate ids in their ranking")
    if has_scores:
        unsorted = sum(any(a < b for a, b in zip(r[1], r[1][1:])) for r in raw_results)
        if unsorted:
            notes.append(f"{unsorted} questions have scores that are not sorted in decreasing order")

    status = "OK" if not problems else "PROBLEM"
    print(f"[{status}] {name}: {n} questions, k={sorted(ks)}, "
          f"{len(retrieved):,} distinct retrieved ids, subset size {len(mapping):,}")
    for m in notes:
        print("   note:", m)
    assert not problems, f"{name}: " + "; ".join(problems)


CHECKS = [  # name, questions, results, mapping, expected_k, has_scores, gold must be in subset
    ("train / random_at60", DATA_FILES["questions"]["train"], DATA_FILES["search"][("train", "random")],
     DATA_FILES["subset"][("train", "random")][1], 60, False, True),
    ("train / Contriever", DATA_FILES["questions"]["train"], DATA_FILES["search"][("train", "contriever")],
     DATA_FILES["subset"][("train", "contriever")][1], 150, True, False),
    ("test / Contriever", DATA_FILES["questions"]["test"], DATA_FILES["search"][("test", "contriever")],
     DATA_FILES["subset"][("test", "contriever")][1], 150, True, False),
    ("test / BM25", DATA_FILES["questions"]["test"], DATA_FILES["search"][("test", "bm25")],
     DATA_FILES["subset"][("test", "bm25")][1], 250, True, False),
    ("test / random (Contriever subset)", DATA_FILES["questions"]["test"], DATA_FILES["search"][("test", "random")],
     DATA_FILES["subset"][("test", "contriever")][1], 60, False, False),
    ("test / random (BM25 subset)", DATA_FILES["questions"]["test"], DATA_FILES["search"][("test", "random")],
     DATA_FILES["subset"][("test", "bm25")][1], 60, False, False),
]

def run_sanity_checks():
    for name, qp, rp, mp, k, scores, gold in CHECKS:
        if not all(Path(p).exists() for p in (qp, rp, mp)):
            print(f"[SKIP] {name}: file missing"); continue
        questions = load_nq(qp)
        raw = pickle.load(open(rp, "rb"))[:len(questions)]   # authors' code uses entry k for question k
        check_data(name, questions, raw, pickle.load(open(mp, "rb")),
                   expected_k=k, has_scores=scores, require_gold_in_subset=gold)

run_sanity_checks()

[OK] train / random_at60: 10000 questions, k=[60, 140], 591,415 distinct retrieved ids, subset size 600,639
   note: list lengths differ: [60, 140]; 1 question(s) off
[OK] train / Contriever: 10000 questions, k=[150], 939,397 distinct retrieved ids, subset size 948,883
[OK] test / Contriever: 2889 questions, k=[150], 357,101 distinct retrieved ids, subset size 1,529,939
[OK] test / BM25: 2889 questions, k=[250], 637,188 distinct retrieved ids, subset size 1,791,311
[OK] test / random (Contriever subset): 2889 questions, k=[60], 172,487 distinct retrieved ids, subset size 1,529,939
[OK] test / random (BM25 subset): 2889 questions, k=[60], 172,487 distinct retrieved ids, subset size 1,791,311


### 1.4 The authors' functions we reuse

Some rules of the experiments live in the authors' code, not in the data. We read `normalize_answers.py`, `prompt_dataset.py` and `read_generation_results.py` and copy only what we need:

- `is_answer_in_text(text, answers)`: a passage "contains the answer" if an answer, or its normalised form (lower case, punctuation, articles and extra spaces removed), is a substring of the normalised text. It selects distracting passages (Table 1) and flags, for each prompt, whether an answer is present in its documents (`ans_in_documents` in their evaluation code).
- `normalize_answer`: the same normalisation, also used for accuracy (Section 4).
- `hash_document(text)`: SHA-256 of a passage, used to drop duplicate passages inside one context.

Their `normalize_text.normalize` is 162 lines of Unicode tables. We copy a shortened version (quotes, dashes, ellipsis), so our matching may differ by a few cases from theirs.

In [14]:
# Future filtering (code from the paper's repo):
def _normalize_unicode(text):            # compact version of the authors' normalize_text.normalize
    text = unicodedata.normalize("NFKC", text)
    for ch in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212\u00ad": text = text.replace(ch, "-")
    for ch in "\u201c\u201d\u201e\u201f\u00ab\u00bb": text = text.replace(ch, '"')
    for ch in "\u2018\u2019\u201a\u201b\u2032\u00b4\u0060": text = text.replace(ch, "'")
    return text.replace("\u2026", "...")

def _white_space_fix(t): return " ".join(t.split())
def _remove_punc(t):
    for p in string.punctuation: t = t.replace(p, " ")
    return t
def _remove_articles(t): return regex.sub(r"\b(a|an|the)\b", " ", t)

def normalize_answer(s, lowercase=True):
    if lowercase: s = s.lower()
    return _white_space_fix(_remove_articles(_remove_punc(_normalize_unicode(s))))

def is_answer_in_text(text, answers):    # authors' rule (normalize_answers.py)
    norm_text = _white_space_fix(_remove_punc(text))
    for a in answers:
        if a in text or normalize_answer(a) in norm_text or normalize_answer(a, False) in norm_text:
            return True
    return False

def hash_document(text): return hashlib.sha256(text.encode()).hexdigest()

### 1.5 Observations that shape the experiments

Each point states whether it was **measured** by running code on the files (cells below), or **taken from the authors' code or README**. The rules that build a prompt are in Section 2.

**Identifiers and subsets**
- **Corpus passages carry no id.** A passage in `processed/*.json` has only `text` and `title`. Its id in the full corpus comes from the mapping. *Measured.*
- **Search results use full-corpus ids, not subset positions.** In the Contriever train subset (948,883 passages), all 939,397 distinct retrieved ids are values of the mapping, and the largest one (13,999,979) is above any subset position. *Measured.*
- **The Contriever train subset is the retrieved passages plus the golds.** The 9,486 passages that no ranking returns are exactly the gold passages. The 10,000 questions have only 9,486 distinct gold passages, so some questions share a gold, and all of them are in the subset. *Measured.*
- **The `random_at60` subset is the random passages plus the golds.** Random ids are drawn over the whole corpus (from 83 to 21,035,205, so the gold zone is included): the question's own gold is never among its random passages (0 / 10,000), but 262 golds of other questions are. *Measured.*
"- **The Table 3 subsets are much larger than what the rankings return** A corpus subset contains all the passages that an experiment might include in the prompt. For Table 3, these are the retrieved passages (Contriever's top 150 for 2,889 questions: 357,101 distinct passages) plus randomly selected passages. For Contriever there are 1,172,838 additional passages (1,529,939 in total) and for BM25, 1,154,123 (1,791,311 in total). We assume it is the random pool.

**Gold passages and the rankings**
- **Gold ids lie at the end of the corpus.** In the test split they go up to 21,035,235, the corpus size minus one. In the train sample the largest is 21,027,658. That's normal: the sample contains only 10,000 of the 72,209 questions in the set, and the “golds” from the other questions take up the rest of the space. *Measured.*
- **No ranking ever contains a gold.** All ids returned by Contriever are below 14,000,000, and the gold id is never in its own top-150 (train 0 / 10,000, test 0 / 2,889). Even the gold *text* is absent under another id (0 / 200 questions checked). So the stored rankings never hold the passage that the annotators used. *Measured.*
- **The gold text is identical in the question file and in the corpus** (200 / 200 checked), so only the title must be read from the corpus. *Measured.*
- **An answer is present in the first four retrieved passages for 72 of 200 questions (36%)**, with the authors' matching rule (shortened normalisation, train questions, Contriever). Accuracy in Table 3 (0.19 to 0.25 with 4 passages) cannot exceed this presence rate, and it is consistent with it. *Measured.*

**Format of the files**
- **Train and test are paired by position.** Entry *k* of a result file refers to question *k* of the same split. All files have matching lengths (10,000 train, 2,889 test).
- **Scores are sorted** in decreasing order in every Contriever and BM25 file (`check_data`).
- **One anomaly.** In `10k_random_results_at60.pkl`, question 5445 has 140 ids instead of 60. The integer after each list is not documented.

**What the authors' code adds (not measured here, used in Section 2)**
- The gold id is not in the stored rankings, so `PromptDataset` inserts it at `gold_position` before reading the passages.
- For Table 1, the ranking is scanned in order. Passages that contain an answer are dropped, duplicates (same text) are dropped, and the first *k* remaining ones are kept. *Authors' code, `_get_answerless_documents_from_indices`.*
- The question files have no title. The prompt format is `Document [id](Title: ...) text`, so we should includes the title. The title comes from the corpus subset and the id is the full-corpus id of the passage. *README (question format) and authors' code.*
- A prompt of at least `max_tokenized_length` tokens is excluded,  so a configuration can end with fewer than 500 questions. *Authors' code, `process_file_data`.*


In [15]:
# Structure of a corpus passage
with open("data/processed/corpus_with_random_at60.json", "rb") as f:
    first = next(ijson.items(f, "item"))
print("fields of a corpus passage:", list(first.keys()))

fields of a corpus passage: ['text', 'title']


In [16]:
# Search results use identifiers from the entire corpus 
mapping = pickle.load(open("data/mappings/subset_to_full_contriever_at150_in_corpus.pkl", "rb"))
full_ids, positions = set(mapping.values()), set(mapping.keys())
res = load_search_results("data/contriever_search_results_at150.pkl")
retrieved = {i for r in res for i in r}
qs_train = load_nq("data/10k_train_dataset.json")
gold_ids = {int(q["idx_gold_in_corpus"]) for q in qs_train}

print("subset size:", len(mapping), "| largest retrieved id:", max(retrieved))
print("retrieved ids found among full ids:", len(retrieved & full_ids), "/", len(retrieved))
print("retrieved ids that are only positions:", len((retrieved & positions) - full_ids))
extra = full_ids - retrieved          # passages in the subset that no ranking returned
print("passages in the subset but never retrieved:", len(extra),
      "| of which gold passages:", len(extra & gold_ids))
print(len(gold_ids), len(gold_ids - full_ids))

subset size: 948883 | largest retrieved id: 13999979
retrieved ids found among full ids: 939397 / 939397
retrieved ids that are only positions: 0
passages in the subset but never retrieved: 9486 | of which gold passages: 9486
9486 0


In [17]:
# Ids range, gold and results
def check_id_ranges(split, questions_path, results_path):
    qs  = load_nq(questions_path)
    res = load_search_results(results_path)
    gold = [int(q["idx_gold_in_corpus"]) for q in qs]
    flat = [i for r in res for i in r]
    print(f"[{split}] questions={len(qs)}  result lists={len(res)}  ids per list={set(map(len, res))}")
    print(f"[{split}] gold ids: min={min(gold):,}  max={max(gold):,}  (max + 1 = {max(gold)+1:,}; corpus size in README: 21,035,236)")
    print(f"[{split}] retrieved ids: max={max(flat):,}  | retrieved ids >= smallest gold id: {sum(i >= min(gold) for i in flat)}")
    print(f"[{split}] gold id inside its own ranking: {sum(g in set(r) for g, r in zip(gold, res))}/{len(qs)}")

check_id_ranges("train", "data/10k_train_dataset.json", "data/contriever_search_results_at150.pkl")
check_id_ranges("test",  "data/test_dataset.json",      "data/contriever_test_search_results_at150.pkl")

[train] questions=10000  result lists=10000  ids per list={150}
[train] gold ids: min=20,970,735  max=21,027,658  (max + 1 = 21,027,659; corpus size in README: 21,035,236)
[train] retrieved ids: max=13,999,979  | retrieved ids >= smallest gold id: 0
[train] gold id inside its own ranking: 0/10000
[test] questions=2889  result lists=2889  ids per list={150}
[test] gold ids: min=20,970,744  max=21,035,235  (max + 1 = 21,035,236; corpus size in README: 21,035,236)
[test] retrieved ids: max=13,999,979  | retrieved ids >= smallest gold id: 0
[test] gold id inside its own ranking: 0/2889


In [18]:
# Gold text identical in the question file and in the corpus
def norm(s): return " ".join(s.split()).lower()

qs = qs_train[:200]
needed = {int(q["idx_gold_in_corpus"]) for q in qs}
corpus_r = load_corpus_subset("data/processed/corpus_with_random_at60.json",
                              "data/mappings/subset_to_full_random_at60_in_corpus.pkl", needed)
same = sum(norm(corpus_r[int(q["idx_gold_in_corpus"])]["text"]) == norm(q["text"]) for q in qs)
print(f"gold text identical in the question file and in the corpus: {same}/{len(qs)}")

gold text identical in the question file and in the corpus: 200/200


In [19]:
# Does the gold text appear in the top-150 retrieved passages under another id? Does an answer appear in the top-4 retrieved passages?
N = 200
qs, res_n = qs_train[:N], res[:N]
needed = {i for r in res_n for i in r}
corpus_c = load_corpus_subset("data/processed/corpus_with_contriever_at150.json",
                              "data/mappings/subset_to_full_contriever_at150_in_corpus.pkl", needed)

twin_rank, ans_in_top4 = [], 0
for q, r in zip(qs, res_n):
    g = norm(q["text"])
    ranks = [k for k, i in enumerate(r) if norm(corpus_c[i]["text"]) == g]
    twin_rank.append(ranks[0] if ranks else None)
    ans_in_top4 += is_answer_in_text(" ".join(corpus_c[i]["text"] for i in r[:4]), q["answers"])

found = [k for k in twin_rank if k is not None]
print(f"gold text found among the top-150 under another id: {len(found)}/{N}")
if found: print("median rank of that twin:", sorted(found)[len(found)//2])
print(f"an answer appears in the top-4 retrieved passages: {ans_in_top4}/{N}")

gold text found among the top-150 under another id: 0/200
an answer appears in the top-4 retrieved passages: 72/200


In [20]:
# Length of all files (train and test must not be mixed)
for name, qp, rp, _, _, _, _ in CHECKS:
    if Path(qp).exists() and Path(rp).exists():
        print(f"{name:22s} questions={len(load_nq(qp)):6d}  result lists={len(pickle.load(open(rp, 'rb'))):6d}")

train / random_at60    questions= 10000  result lists= 10000
train / Contriever     questions= 10000  result lists= 10000
test / Contriever      questions=  2889  result lists=  2889
test / BM25            questions=  2889  result lists=  2889
test / random (Contriever subset) questions=  2889  result lists= 10000
test / random (BM25 subset) questions=  2889  result lists= 10000


In [21]:
# Random passages: where do they come from, and is the question's own gold ever among them?
rnd = load_search_results("data/10k_random_results_at60.pkl")
flat = [i for r in rnd for i in r]
gold_list = [int(q["idx_gold_in_corpus"]) for q in qs_train]
print("random ids range:", min(flat), "to", max(flat))
print("own gold among its random passages:", sum(g in set(r) for g, r in zip(gold_list, rnd)), "/", len(rnd))
print("golds of other questions drawn as random passages:", len(set(flat) & set(gold_list)))
m = pickle.load(open("data/mappings/subset_to_full_random_at60_in_corpus.pkl", "rb"))
print("subset size:", len(m), "| random ids + golds:", len(set(flat) | set(gold_list)))

random ids range: 83 to 21035205
own gold among its random passages: 0 / 10000
golds of other questions drawn as random passages: 262
subset size: 600639 | random ids + golds: 600639


In [22]:
# Composition of the test subsets: what is in them besides the retrieved passages?
test_qs = load_nq("data/test_dataset.json")
test_gold = {int(q["idx_gold_in_corpus"]) for q in test_qs}
gold_min = min(test_gold)
for name, res_path, map_path in [
    ("Contriever", "data/contriever_test_search_results_at150.pkl",
     "data/mappings/subset_to_full_test_random_contriever_in_corpus.pkl"),
    ("BM25", "data/bm25_test_search_results_at250.pkl",
     "data/mappings/subset_to_full_test_random_bm25_in_corpus.pkl")]:
    subset = set(pickle.load(open(map_path, "rb")).values())
    ret = {i for r in load_search_results(res_path) for i in r}
    extra = subset - ret
    print(f"{name}: subset={len(subset):,} retrieved={len(ret):,} extra={len(extra):,} | "
          f"extra below 14M: {sum(i < 14_000_000 for i in extra):,} | "
          f"extra in gold zone: {sum(i >= gold_min for i in extra):,} | "
          f"test golds in subset: {len(test_gold & subset)}/{len(test_gold)} | "
          f"id range of extras: {min(extra):,} to {max(extra):,}")

Contriever: subset=1,529,939 retrieved=357,101 extra=1,172,838 | extra below 14M: 767,519 | extra in gold zone: 13,895 | test golds in subset: 2841/2841 | id range of extras: 37 to 21,035,235
BM25: subset=1,791,311 retrieved=637,188 extra=1,154,123 | extra below 14M: 763,621 | extra in gold zone: 9,777 | test golds in subset: 2841/2841 | id range of extras: 37 to 21,035,226


### 1.6 What we could not observe, and why

- **The authors' retrieval index.** The FAISS index is not distributed, so we cannot tell whether the gold passages were indexed, or whether the original passage was removed from the corpus before indexing. Both would explain why no gold text appears in any ranking. Our only evidence is indirect: no gold id appears in any ranking.
- **The full corpus.** We download only the subsets (the full file is about 12 GB), so we cannot look for a passage outside them.
- **How the subsets were built and how random passages were drawn** (uniform over the corpus? gold excluded?). This is not documented.
- **How the Table 3 random passages were drawn.** About 0.8 to 1.2% (BM25 / Contriever) of the extra passages lie in the gold zone (0.3% expected for a uniform draw over the corpus). The README mentions "random entries from the NQ dataset", but we did not establish the sampling rule.
- **The integer after each list of ids in `10k_random_results_at60.pkl`.** It increases with the question (0, 12, 19, ...) and is different for each of the 10,000 questions. We do not know what it is and we do not use it.
- **How many prompts exceed the context length.** The filtering happens when prompts are built (Section 2), not in the files. The number of questions skipped per configuration is known only after the prompts are built. In their code, each prompt is constructed and then tokenized. If it reaches max_tokenized_length tokens, the question is excluded from this configuration. As a result, two test configurations may end up with different sets of questions. For paired tests, comparisons should be made only between the questions that appear in both tests.
- **Our matching rules against theirs.** We copy a shortened version of their Unicode normalisation (Section 1.4), so counts such as 72 / 200 may differ slightly from what their code would give.
- **The effect of the title on accuracy.** We include the title to follow the authors' prompt but did not test its effect.

## 2. Prompts

Layout (paper, Figure 1): task instruction, then `Documents:`, then the documents, then `Question:` and `Answer:`. The question is always last.

**Verified in the authors' code (`PromptDataset`)**
- The instruction string and the layout `{instruction}\nDocuments:\n{documents}\nQuestion: {query}\nAnswer:`.
- The number in `Document [i]` is the passage's id in the full corpus, not its position in the context.
- Distracting passages (Table 1): scan the retrieved list in order, drop passages that contain an answer or duplicate an earlier text, keep the first *k*, then insert the gold at `gold_position`.
- Prompts of at least `max_tokenized_length` tokens are skipped. Two configurations can therefore end with different sets of questions, so paired comparisons use only the questions present in both.
- For closed-book, the prompt instruction is different: "You are given a question and you MUST respond with a short answer (max 5 tokens) based on your internal knowledge. If you do not know the answer, please respond with NO-RES." then, "\nQuestion: ...\nAnswer:" (without "Documents:").
- The scripts pass `do_normalize_query=True`, so the query is cleaned up by `normalize_text.normalize` (Unicode quotes and hyphens) before it reaches the prompt.

**Still to verify**: whether the Llama-2 chat template is applied at generation (`generate_answers_llm.py`), and how random passages are drawn (the data show they are drawn from the whole corpus, golds included, never the question's own gold).

In [ ]:
INSTRUCTION = (
    "You are given a question and you MUST respond by EXTRACTING the answer (max 5 tokens) "
    "from one of the provided documents. If none of the documents contain the answer, "
    "respond with NO-RES."
)   # from Figure 1 of the paper; [TO VERIFY] against the code

def format_document(number: int, title: str, text: str) -> str:
    """One document as in the paper: `Document [number](Title: title) text`.
    TODO.
    """
    raise NotImplementedError


def build_prompt(question: str, docs: list[dict], instruction: str = INSTRUCTION) -> str:
    """Instruction, `Documents:`, the documents in the given order, then the question.
    `docs` is a list of {"title", "text"}; an empty list gives the closed-book prompt
    (question only, no `Documents:` block). TODO.
    """
    raise NotImplementedError

In [ ]:
def sample_random_docs(corpus: dict[int, dict], n: int, exclude: set[int], rng: random.Random) -> list[dict]:
    """n random passages from the corpus subset, never in `exclude`
    (gold + the question's retrieved passages). [TO VERIFY] how the authors drew them.
    TODO.
    """
    raise NotImplementedError


def get_distractors(q: dict, search_results: dict, corpus: dict[int, dict], n: int) -> list[dict]:
    """Top-ranked passages of the retriever that do NOT contain any reference answer,
    in rank order. Table 1 setting. TODO.
    """
    raise NotImplementedError


def context_oracle(gold: dict, others: list[dict], gold_position: str = "near") -> list[dict]:
    """gold + others. "near": gold last (next to the question); "far": gold first.
    TODO.
    """
    raise NotImplementedError


def context_mixed(retrieved: list[dict], random_docs: list[dict]) -> list[dict]:
    """Table 3 layout: random documents first, retrieved documents next to the question.
    [TO VERIFY] order of the retrieved block (rank 1 first or last).
    TODO.
    """
    raise NotImplementedError

In [ ]:
def test_prompt_falcon() -> None:
    """Build the Figure 1 example by hand, print it, and compare it with the paper."""
    gold = {"title": "Millennium Falcon",
            "text": "Han Solo won the Millennium Falcon from Lando Calrissian in the card game sabacc..."}
    prompt = build_prompt("who owned the millennium falcon before han solo", [gold])
    print(prompt)
    assert prompt.startswith(INSTRUCTION)
    assert "Title: Millennium Falcon" in prompt
    assert prompt.rstrip().endswith("Answer:")
    # TODO: add a closed-book check and a "near" vs "far" ordering check

# test_prompt_falcon()

## 3. Generation

Llama-2-7B-Chat in 4 bits, greedy decoding, 15 new tokens. Batched generation with a decoder-only model needs **left padding**. Log the number of prompt tokens and flag prompts longer than the context window (never truncate silently).

In [ ]:
def load_model(model_id: str):
    """Return (model, tokenizer). 4-bit with bitsandbytes (nf4 or fp4: [TO VERIFY] the paper's choice),
    fp16 compute, device_map="auto". Tokenizer: padding_side="left", set a pad token.
    TODO.
    """
    raise NotImplementedError


def generate(prompts: list[str], model, tokenizer, batch_size: int, max_new_tokens: int) -> list[dict]:
    """Greedy generation. Return, for each prompt, {"response": str (new tokens only),
    "prompt_tokens": int, "output_tokens": int, "hit_cap": bool}. TODO.
    """
    raise NotImplementedError

## 4. Evaluation

Paper metric: an output is correct if it contains at least one of the reference answers. The paper acknowledges that this is crude (e.g. "Roosevelt" vs "President Roosevelt"). Port the authors' function (`read_generation_results.py`) first, then document any difference.

Accuracy in the authors’ code (are_answers_matching): the output is correct if at least one normalised reference answer is a substring of the normalised output; normalisation lower-cases, removes punctuation and articles and collapses spaces (Section 1.4). » Cela règle le [TO VERIFY] de contains_answer.

In [ ]:
def contains_answer(response: str, answers: list[str]) -> bool:
    """Paper accuracy for one question. [TO VERIFY] normalisation in the original code
    (lower-casing, punctuation, articles). TODO.
    """
    raise NotImplementedError

## 5. Experiments

An `ExperimentSpec` describes one configuration. `run_experiment` builds all prompts for the sampled questions, generates, scores, and writes `outputs/nq/<name>.jsonl` (one line per question, with a field `correct` in {0, 1}). If the file is complete, it is skipped.

Table 1 and Table 2 use a sample of the 10K train sample (`split="train"`); Table 3 uses the test split.

In [ ]:
@dataclass
class ExperimentSpec:
    name: str
    kind: str                      # "closed_book" | "oracle" | "mixed"
    split: str                     # "train" (Tables 1-2) | "test" (Table 3)
    retriever: Optional[str] = None
    n_distracting: int = 0         # oracle, Table 1
    n_random: int = 0              # oracle (Table 2) or mixed (Table 3)
    n_retrieved: int = 0           # mixed
    gold_position: str = "near"    # oracle: "near" | "far"


def build_experiment_grid() -> list[ExperimentSpec]:
    specs = [ExperimentSpec("closed_book", "closed_book", "train"),
             ExperimentSpec("gold_only", "oracle", "train")]
    for n in (1, 2, 4):                                   # Table 1
        specs.append(ExperimentSpec(f"t1_dist{n}", "oracle", "train", retriever="contriever", n_distracting=n))
    for n in (2, 4):                                      # Table 2
        specs.append(ExperimentSpec(f"t2_rand{n}", "oracle", "train", n_random=n))
    for r in ("contriever", "bm25"):                      # Table 3
        for n_ret in (1, 4):
            for n_rand in (0, 3, 10):
                specs.append(ExperimentSpec(f"t3_{r}_ret{n_ret}_rand{n_rand}", "mixed", "test",
                                            retriever=r, n_retrieved=n_ret, n_random=n_rand))
    return specs

GRID = build_experiment_grid()
print(len(GRID), "configurations")

In [ ]:
def build_prompts_for_spec(spec: ExperimentSpec, questions: list[dict], corpus: dict, search_results: dict,
                           rng: random.Random) -> list[str]:
    """One prompt per question for this configuration, using the helpers of section 2.
    Use a per-question RNG derived from (seed, example_id) so that random documents do not
    depend on the order in which configurations are run. TODO.
    """
    raise NotImplementedError


def run_experiment(spec: ExperimentSpec, questions: list[dict], corpus: dict, search_results: dict,
                   model, tokenizer, out_dir: Path) -> Path:
    """Resume-safe runner.
      - output file: out_dir / f"{spec.name}.jsonl"; skip questions already written (key: example_id);
      - append a line after each batch: example_id, question, answers, response, correct,
        prompt_tokens, output_tokens, hit_cap, plus the spec fields;
      - log the time per batch.
    TODO.
    """
    raise NotImplementedError

## 6. Validation runs (before any grid)

Run the first two configurations on 20, then 100 questions, and compare with `REFERENCE`.
Do not go further if closed-book or gold-only is far from the paper.

In [ ]:
# TODO: load data (section 1), load the model (section 3), then:
# for spec in GRID[:2]:
#     run_experiment(spec, questions, corpus, search_results, model, tokenizer, CONFIG["output_dir"])
# then compare with REFERENCE["closed_book"] and REFERENCE["gold_only"] (section 7)

## 7. Full grids

Switch `CONFIG["debug"]` off and run in this order: Table 1, Table 2, Table 3. Time one configuration first; the whole reproduction is ~9,500 generations (estimated 3 to 7 hours).

In [ ]:
# for spec in GRID[2:]:
#     run_experiment(spec, ...)

## 8. Analysis

Per-query scores are kept so that every comparison is **paired** (same questions in both conditions).

In [ ]:
def load_results(out_dir: Path) -> pd.DataFrame:
    """Concatenate all .jsonl files into one DataFrame (one row per question and configuration).
    TODO.
    """
    raise NotImplementedError


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """Accuracy per configuration, with n, a 95% bootstrap CI, and the paper's value next to it
    (from REFERENCE) and the difference. TODO.
    """
    raise NotImplementedError


def paired_comparison(df: pd.DataFrame, cond_a: str, cond_b: str, n_boot: int = 10_000, seed: int = 0) -> dict:
    """Compare two configurations on the questions they share. Return
    {"n", "mean_a", "mean_b", "delta", "wilcoxon_p", "ttest_p", "ci95": (low, high)}.
    Hints: scipy.stats.wilcoxon, scipy.stats.ttest_rel, bootstrap over questions.
    TODO.
    """
    raise NotImplementedError

In [ ]:
# Comparisons to report (trends, not only values):
COMPARISONS = [
    ("gold_only", "t1_dist1"), ("gold_only", "t1_dist2"), ("gold_only", "t1_dist4"),   # distracting hurt?
    ("gold_only", "t2_rand2"), ("gold_only", "t2_rand4"),                              # random help?
    ("t3_contriever_ret4_rand0", "t3_contriever_ret4_rand10"),                         # heuristic, Contriever
    ("t3_bm25_ret4_rand0", "t3_bm25_ret4_rand10"),                                     # heuristic, BM25
]

## 9. What worked, what did not

TODO once results are in:
- Is the direction of each effect reproduced? Which comparisons are significant (Wilcoxon p < 0.01, paired t-test, bootstrap CI)?
- Where do we differ from the paper (sample of 500, quantization, prompt details, answer matching)?
- Output lengths and truncation rate (15-token cap): do they explain gaps?
- Anything that failed, with a diagnosis (a documented failed reproduction is a valid result).